In [ ]:
# Databricks notebook source
# 06_gold — Intermediate findings -> Risk Card core
# Contract: docs/contracts/poc-intermediate-gold-schema.md
# Eval is intentionally not read by this notebook.

In [ ]:
dbutils.widgets.text("catalog", "access_drift")
dbutils.widgets.text("run_id", "poc")

CATALOG = dbutils.widgets.get("catalog")
RUN_ID = dbutils.widgets.get("run_id") or "poc"

In [ ]:
from pyspark.sql import functions as F

findings = spark.table(
    f"{CATALOG}.intermediate.nhi_residual_access_findings"
).alias("finding")
principals = spark.table(f"{CATALOG}.silver.silver_principals")
assets = spark.table(f"{CATALOG}.silver.silver_assets")

subjects = principals.alias("subject")
owners = principals.alias("owner")
asset_nodes = assets.alias("asset")

joined = (
    findings
    .join(subjects, F.col("finding.subject_principal_id") == F.col("subject.principal_id"), "inner")
    .join(owners, F.col("finding.owner_principal_id") == F.col("owner.principal_id"), "left")
    .join(asset_nodes, F.col("finding.asset_id") == F.col("asset.asset_id"), "inner")
)

subject_name = F.coalesce(
    F.col("subject.display_name"),
    F.col("subject.source_principal_id"),
    F.col("finding.subject_source_principal_id"),
)
owner_name = F.coalesce(
    F.col("owner.display_name"),
    F.col("owner.email"),
    F.col("owner.source_principal_id"),
    F.col("finding.owner_source_principal_id"),
)
asset_name = F.coalesce(
    F.col("asset.display_name"), F.col("finding.asset_source_id")
)
severity = F.upper(F.col("finding.asset_sensitivity"))
subject_label = (
    F.when(F.col("finding.principal_type") == F.lit("github_app"), F.lit("GitHub App"))
     .when(F.col("finding.principal_type") == F.lit("app_registration"), F.lit("App Registration"))
     .when(F.col("finding.principal_kind") == F.lit("HI"), F.lit("User"))
     .otherwise(F.col("finding.principal_type"))
)
entra_nhi_evidence_summary = F.concat(
    F.lit("퇴사자 "), owner_name,
    F.lit("가 소유한 활성 "), subject_label, F.lit(" "), subject_name,
    F.lit("에 유효한 credential이 남아 있고 "), asset_name,
    F.lit(" 자산 접근 경로가 활성 상태입니다."),
)
github_nhi_evidence_summary = F.concat(
    F.lit("owner 이메일 "), owner_name,
    F.lit("이 연결된 활성 GitHub App "), subject_name,
    F.lit("에 유효한 credential이 남아 있고 "), asset_name,
    F.lit(" repository 접근 경로가 활성 상태입니다."),
)
hi_evidence_summary = F.concat(
    F.lit("퇴사 또는 비활성 사용자 "), subject_name,
    F.lit("에게 "), asset_name,
    F.lit(" 자산 접근 권한이 활성 상태로 남아 있습니다."),
)
evidence_summary = (
    F.when(F.col("finding.principal_kind") == F.lit("HI"), hi_evidence_summary)
     .when(F.col("finding.principal_type") == F.lit("github_app"), github_nhi_evidence_summary)
     .otherwise(entra_nhi_evidence_summary)
)
recommended_action = (
    F.when(
        F.col("finding.principal_kind") == F.lit("HI"),
        F.lit(
            "사용자 상태와 업무 필요성을 확인한 뒤 남아 있는 SaaS 자산 접근 권한을 회수하거나 최소 권한으로 조정합니다."
        ),
    )
    .when(
        F.col("finding.principal_type") == F.lit("github_app"),
        F.lit(
            "현재 담당자와 GitHub App 사용 여부를 확인한 뒤 deploy key/PAT를 rotation 또는 revoke하고 "
            "불필요한 repository 권한을 회수합니다."
        ),
    )
    .otherwise(F.lit(
        "현재 담당자와 사용 여부를 확인한 뒤 credential을 rotation 또는 revoke하고 "
        "불필요한 자산 권한을 회수합니다."
    ))
)

gold = joined.select(
    F.col("finding.risk_case_key"),
    F.col("finding.finding_id"),
    F.col("finding.drift_type"),
    F.col("finding.subject_type"),
    F.col("finding.principal_kind"),
    F.col("finding.source_system"),
    F.col("finding.principal_type"),
    F.col("finding.finding_type_label"),
    F.col("finding.risk_factors"),
    F.col("finding.subject_principal_id"),
    F.col("finding.subject_source_principal_id"),
    subject_name.alias("subject_display_name"),
    F.col("finding.owner_principal_id"),
    owner_name.alias("owner_display_name"),
    F.col("finding.asset_id"),
    F.col("finding.asset_source_id"),
    asset_name.alias("asset_display_name"),
    F.col("finding.asset_environment"),
    F.col("finding.asset_sensitivity"),
    severity.alias("severity"),
    evidence_summary.alias("evidence_summary"),
    recommended_action.alias("recommended_action"),
    F.col("finding.owner_edge_id"),
    F.col("finding.access_edge_id"),
    F.col("finding.credential_id"),
    F.col("finding.access_last_used_at").alias("last_accessed_at"),
    F.col("finding.detected_at"),
    F.lit(RUN_ID).alias("run_id"),
)

In [ ]:
target = f"{CATALOG}.gold.gold_core"

classification_columns = {
    "principal_kind": "STRING",
    "source_system": "STRING",
    "principal_type": "STRING",
    "finding_type_label": "STRING",
    "risk_factors": "ARRAY<STRING>",
    "last_accessed_at": "TIMESTAMP",
}
existing_columns = set(spark.table(target).columns)
missing_classification_columns = [
    column for column in classification_columns
    if column not in existing_columns
]
if missing_classification_columns:
    add_columns_sql = ", ".join(
        f"{column} {classification_columns[column]}"
        for column in missing_classification_columns
    )
    spark.sql(f"ALTER TABLE {target} ADD COLUMNS ({add_columns_sql})")
    print(f"Added classification columns to {target}: {missing_classification_columns}")

nullable_evidence_columns = ["credential_id", "owner_edge_id"]
for column in nullable_evidence_columns:
    try:
        spark.sql(f"ALTER TABLE {target} ALTER COLUMN {column} DROP NOT NULL")
    except Exception as exc:
        message = str(exc).lower()
        if "not null" not in message and "nullable" not in message:
            raise

target_columns = spark.table(target).columns
gold_columns = gold.columns

missing_columns = [column for column in target_columns if column not in gold_columns]
extra_columns = [column for column in gold_columns if column not in target_columns]
if missing_columns or extra_columns:
    raise ValueError(
        f"schema mismatch for {target}: missing={missing_columns}, extra={extra_columns}"
    )

if gold_columns != target_columns:
    print(f"Aligning {target} columns to target table order")
    print(f"dataframe columns: {gold_columns}")
    print(f"target columns: {target_columns}")

gold_aligned = gold.select(*target_columns)
gold_aligned.write.format("delta").mode("overwrite").saveAsTable(target)

result = spark.table(target)
count = result.count()
assert count >= 1, "expected at least one Gold Core row"

duplicate_keys = result.groupBy("risk_case_key").count().where("count > 1").count()
assert duplicate_keys == 0, f"duplicate Gold risk_case_key rows: {duplicate_keys}"

intermediate_count = spark.table(f"{CATALOG}.intermediate.nhi_residual_access_findings").count()
assert count == intermediate_count, f"Gold row count mismatch: gold={count}, intermediate={intermediate_count}"

required = [
    "risk_case_key", "finding_id", "principal_kind", "source_system",
    "principal_type", "finding_type_label", "risk_factors",
    "subject_display_name", "owner_display_name", "asset_display_name",
    "severity", "evidence_summary", "recommended_action",
]
missing_required = result.where(
    F.expr(" OR ".join(f"{column} IS NULL" for column in required))
).count()
assert missing_required == 0, f"Gold rows with missing Risk Card fields: {missing_required}"

nhi_missing_evidence = result.where(
    "principal_kind = 'NHI' AND credential_id IS NULL"
).count()
assert nhi_missing_evidence == 0, f"NHI Gold rows with missing credential evidence: {nhi_missing_evidence}"

nhi_missing_owner_evidence = result.where(
    "principal_kind = 'NHI' AND owner_edge_id IS NULL"
).count()
assert nhi_missing_owner_evidence == 0, f"NHI Gold rows with missing owner edge evidence: {nhi_missing_owner_evidence}"

expected_sources = {"github", "google_workspace", "slack"}
actual_sources = {row["source_system"] for row in result.select("source_system").distinct().collect()}
missing_sources = expected_sources - actual_sources
assert not missing_sources, f"missing expected SaaS source_system values in Gold: {sorted(missing_sources)}"

expected_labels = {
    "NHI Access Drift",
    "NHI Credential Drift",
    "NHI Permission Drift",
    "HI Access Drift",
    "HI Permission Drift",
    "HI Stale Access",
}
actual_labels = {row["finding_type_label"] for row in result.select("finding_type_label").distinct().collect()}
missing_labels = expected_labels - actual_labels
assert not missing_labels, f"missing expected finding_type_label values in Gold: {sorted(missing_labels)}"

result.select(
    "risk_case_key", "finding_type_label", "principal_kind", "source_system",
    "principal_type", "subject_display_name", "owner_display_name",
    "asset_display_name", "severity", "evidence_summary", "recommended_action",
).orderBy("subject_display_name").show(truncate=False)
print(f"gold_core: {count} rows")
print("gold smoke test passed")